---
title: Sampler quickstart
description: How to use the Sampler primitive in Qiskit Runtime.
---


# Sampler quickstart

The Sampler's core task is sampling the output register from the execution of one or more quantum circuits. [Dynamic circuits](/docs/guides/execute-dynamic-circuits) and parameterized circuits are accepted as input (if parametrized circuits are submitted, the parameter values must also be provided). Sampler also supports built-in dynamical decoupling and twirling for [error suppression](/docs/guides/error-mitigation-and-suppression-techniques).

The steps in this topic describe how to set up Sampler, explore the options you can use to configure it, and invoke it in a program.

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.5.0
qiskit-ibm-runtime~=0.47.0
```
</AccordionItem>
</Accordion>

## Steps to use the Sampler primitive
### 1. Initialize the account

Because Qiskit Runtime is a managed service, you first need to initialize your account. You can then select the QPU you want to use to calculate the expectation value.

Follow the steps in the [Set up your IBM Cloud account](/docs/guides/cloud-setup) topic if you don't already have an account set up.

<Admonition type="note" title="Fractional gates">
    To use the newly supported [fractional gates](/docs/guides/fractional-gates), set `use_fractional_gates=True` when requesting a backend from a `QiskitRuntimeService` instance. For example:
    ```python
    service = QiskitRuntimeService()
    fractional_gate_backend = service.least_busy(use_fractional_gates=True)
    ```

    This is an experimental feature and might change in the future.
</Admonition>

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

### 2. Create a circuit

You need at least one circuit as the input to the Sampler primitive.

In [2]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(127, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

The circuit and observable need to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 3036), ('sx', 1769), ('cz', 378), ('measure', 127), ('barrier', 1)])


### 3. Initialize the Qiskit Runtime Sampler

When you initialize Sampler, use the `mode` parameter to specify the mode you want it to run in.  Possible values are `batch`, `session`, or `backend` objects for batch, session, and job execution mode, respectively. For more information, see [Introduction to Qiskit Runtime execution modes.](execution-modes) Note that Open Plan users cannot submit session jobs.

In [4]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

### 4. Invoke Sampler and get results

Next, invoke the `run()` method to generate the output. The circuit and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d9bjioe6hjac73ffpt00


>>> Job Status: QUEUED


In [6]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(
    f"First ten results for the 'meas' output register: "
    f"{pub_result.data.meas.get_bitstrings()[:10]}"
)

First ten results for the 'meas' output register: ['1101010100010000110010011111001100111000110001001100000001111010010000110001000110011011001000011101010101011011011001100100011', '1010000011001001001010000000010011101110010000011010001001000010110000100010011010101110001100000100000011011111001000000000001', '1001100110001100100011000001000010000010011010100000000011010011110101001100001101100000001010000111010000011010010001100110011', '0101010101010101010100111011101110010110010101110000100010100010110011000010101000000110011001011101010000101011101001100001000', '0101011011101110110010101100100011111001001100001101001001101010000000110101001010011000010100100011000001000101010001100010011', '1101010101001111011111100111001001111110011010101000011100001110000101111010000011100110101010001101000011001100110011001010000', '0000000000000010001000111111100011101100011010101110011011110011100110011001010100011100101100100011100110011111001000110010000', '1010101000100110110100010100000

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [test locally](local-testing-mode) before running on quantum computers.
    - Review detailed [examples](sampler-examples).
    - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
    - Learn how to transpile locally in the [Transpile](transpile/) section.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings#compare-transpiler-settings) guide.
    - Learn how to [use the primitive options](/docs/guides/runtime-options-overview).
    - View the API for [Sampler](/docs/api/qiskit-ibm-runtime/options-sampler-options) options.
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>